# 02. 収集データのレビュー

`01_collect_data.ipynb` で集めたデータを可視化して確認します。学習 (`03_train_bc.ipynb`) に進む前に、
意図した通りのデータが集まっているかをここで確認してください。このノートブックは読み取り専用で、
データ収集や学習には影響しません。

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

DATA_DIR = Path("../data")
MANIFEST_PATH = DATA_DIR / "manifest.csv"
EPISODES_DIR = DATA_DIR / "episodes"

FILENAME_RE = re.compile(r"^(\d+)_a(\d+)_")

manifest = pd.read_csv(MANIFEST_PATH)
manifest

## 要約統計

In [ ]:
print(f"エピソード数: {len(manifest)}")
print(f"クリア率 (flag_get): {manifest['flag_get'].mean():.1%}")
manifest[["num_frames", "score", "x_pos_max", "coins", "time_remaining"]].describe()

## アクション分布 (全エピソード合計)

学習時のクラス不均衡を事前に把握するために、7種類のアクションそれぞれが何フレーム記録されたかを集計する。

In [ ]:
from collections import Counter

from gym_super_mario_bros.actions import SIMPLE_MOVEMENT

ACTION_NAMES = ["_".join(a) if a else "NOOP" for a in SIMPLE_MOVEMENT]

action_counts = Counter()
for ep_dir in sorted(EPISODES_DIR.glob("ep_*")):
    for img_path in ep_dir.glob("*.png"):
        m = FILENAME_RE.match(img_path.name)
        if m:
            action_counts[int(m.group(2))] += 1

counts = [action_counts.get(i, 0) for i in range(len(SIMPLE_MOVEMENT))]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(ACTION_NAMES, counts)
ax.set_ylabel("フレーム数")
ax.set_title("アクションの出現回数")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## エピソードのフレームサンプル表示

`episode_id` を変えて、別のエピソードを確認できる。

In [ ]:
episode_id = manifest["episode_id"].iloc[0]  # 確認したいエピソードIDに変更してください
ep_dir = EPISODES_DIR / episode_id

image_paths = sorted(ep_dir.glob("*.png"))
print(f"{episode_id}: {len(image_paths)} フレーム")

n_samples = min(8, len(image_paths))
sample_idx = np.linspace(0, len(image_paths) - 1, num=n_samples, dtype=int)

fig, axes = plt.subplots(1, n_samples, figsize=(3 * n_samples, 3))
if n_samples == 1:
    axes = [axes]
for ax, idx in zip(axes, sample_idx):
    path = image_paths[idx]
    m = FILENAME_RE.match(path.name)
    action_idx = int(m.group(2)) if m else -1
    ax.imshow(Image.open(path))
    ax.set_title(ACTION_NAMES[action_idx] if action_idx >= 0 else "?", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## アクション系列の時系列プロット

選択したエピソード内で、どのタイミングでどのアクションを取ったかを確認できる(ジャンプのタイミング等)。

In [ ]:
action_seq = []
for path in image_paths:
    m = FILENAME_RE.match(path.name)
    action_seq.append(int(m.group(2)) if m else -1)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(action_seq)
ax.set_xlabel("フレーム番号")
ax.set_ylabel("アクション")
ax.set_yticks(range(len(SIMPLE_MOVEMENT)))
ax.set_yticklabels(ACTION_NAMES)
ax.set_title(f"{episode_id} のアクション系列")
plt.tight_layout()
plt.show()

## データのクリーニング

数フレームしかない、記録ミスなど明らかにおかしいエピソードが見つかった場合は、
該当する `data/episodes/ep_XXXX/` フォルダを削除し、`data/manifest.csv` から対応する行を手動で削除してください。